# Session 1 · LLM extension: smoke tests, Binoculars, LLM learning-rate tuning

Settings: **Accelerator GPU T4 x2**, **Internet On**, Input: the `raid-processed` dataset
(new version with `train_llm.py` and `zero_shot_binoculars.py`).
Run with **Save Version → Save & Run All (Commit)**. Expected time: about 5 hours.
If a smoke test fails, the run stops there within minutes.

In [1]:
# Copy scripts and data from the raid-processed dataset into a working project folder
import glob, os, shutil
# (the folder that has both the data and the scripts, i.e. the dataset itself -
#  not a copy inside an earlier notebook's output)
roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train_clean.parquet", recursive=True)]
roots = [r for r in roots if glob.glob(f"{r}/**/train_llm.py", recursive=True)]
assert roots, "raid-processed dataset (with train_llm.py) not found - check the Input panel"
data_root = roots[0]
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/data/processed", exist_ok=True)
for f in glob.glob(f"{data_root}/**/*.py", recursive=True):
    shutil.copy(f, "/kaggle/working/src/")
for f in glob.glob(f"{data_root}/**/*.parquet", recursive=True):
    shutil.copy(f, "/kaggle/working/data/processed/")
os.chdir("/kaggle/working")
print("src:", sorted(os.listdir("src")))
print("data:", sorted(os.listdir("data/processed")))
for need in ["train_llm.py", "zero_shot_binoculars.py", "metrics.py", "train_roberta.py"]:
    assert need in os.listdir("src"), f"{need} missing - upload it to the dataset (New Version)"
!pip uninstall -y -q torchao
!pip install -q peft
!nvidia-smi --query-gpu=name,memory.total --format=csv

src: ['inspect_raid.py', 'make_splits.py', 'metrics.py', 'train_classical.py', 'train_llm.py', 'train_roberta.py', 'zero_shot_binoculars.py']
data: ['test_heldout_attacks.parquet', 'test_reppen.parquet', 'test_seen.parquet', 'test_seen_attacks.parquet', 'test_unseen.parquet', 'train_attacked.parquet', 'train_clean.parquet', 'val.parquet']
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1. Smoke tests (a few minutes each - numbers are meaningless)

In [2]:
!python src/zero_shot_binoculars.py --limit 200 --out results/smoke_binoculars

Observer on cuda:0, performer on cuda:1
Tokenizing...
config.json: 100%|█████████████████████████████| 684/684 [00:00<00:00, 2.21MB/s]
tokenizer_config.json: 100%|███████████████| 7.23k/7.23k [00:00<00:00, 15.3MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 49.2MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 112MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 159MB/s]
  990 texts tokenized
Loading models...

model.safetensors: downloading bytes:                       | 4.65MB            
model.safetensors: downloading bytes: █▍                    |  204MB, 14.7MB/s  
model.safetensors: downloading bytes: ██▊                   |  392MB, 32.2MB/s  
model.safetensors: downloading bytes: ███▌                  |  507MB, 41.8MB/s  
model.safetensors: reconstructing file:  11%|▋     |  338MB / 3.09GB, 25.7MB/s  
model.safetensors: downloading bytes: ███████▌              | 1.07GB, 90.0MB/s  
model.safetens

In [3]:
!python src/train_llm.py --limit 500 --seeds 42 --aug 0 --epochs 1 --out results/smoke_llm

Device: cuda
Tokenizing...
config.json: 100%|█████████████████████████████| 681/681 [00:00<00:00, 2.35MB/s]
tokenizer_config.json: 100%|███████████████| 7.23k/7.23k [00:00<00:00, 15.3MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 49.2MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 113MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 145MB/s]
  21,722 texts tokenized
Run aug=0.0 lr=0.0001 seed=42

model.safetensors: downloading bytes: ██                    | 93.5MB, 6.85MB/s  
model.safetensors: downloading bytes: ██▌                   |  116MB, 10.6MB/s  
model.safetensors: downloading bytes: ███████████           |  496MB, 40.2MB/s  
model.safetensors: downloading bytes: ████████████▏         |  545MB, 43.9MB/s  
model.safetensors: downloading bytes: █████████████▏        |  591MB, 48.0MB/s  
model.safetensors: downloading bytes: ██████████████████▏   |  817MB, 68.3MB/s  
model.safetensors: downloa

## 2. Zero-shot Binoculars, full run (no training; about 1.5-2 h)

In [4]:
!python src/zero_shot_binoculars.py

Observer on cuda:0, performer on cuda:1
Tokenizing...
  56,168 texts tokenized
Loading models...
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 255.07it/s]
val: scoring 4,012 texts
      batch 50/502  19.5 texts/s  ETA 3 min
      batch 100/502  14.6 texts/s  ETA 4 min
      batch 150/502  12.1 texts/s  ETA 4 min
      batch 200/502  10.6 texts/s  ETA 4 min
      batch 250/502  9.6 texts/s  ETA 4 min
      batch 300/502  8.7 texts/s  ETA 3 min
      batch 350/502  7.9 texts/s  ETA 3 min
      batch 400/502  7.3 texts/s  ETA 2 min
      batch 450/502  6.7 texts/s  ETA 1 min
      batch 500/502  6.1 texts/s  ETA 0 min
      batch 502/502  6.1 texts/s  ETA 0 min
test_seen: scoring 4,012 texts
      batch 50/502  18.7 texts/s  ETA 3 min
      batch 100/502  14.3 texts/s  ETA 4 min
      batch 150/502  11.9 texts/s  ETA 4 min
      batch 200/502  10.5 texts/s  ETA 4 min
      batch 250/502  9.6 texts/s  ETA 4 min
      batch 300/502  8.7 texts/s  ETA 3 min
      batch 

## 3. LLM learning-rate tuning on validation only (3 runs, about 2.5-3 h)

In [5]:
!python src/train_llm.py --seeds 42 --aug 0 --lrs 5e-5 1e-4 2e-4 --tune-only --out results/tuning

Device: cuda
Tokenizing...
  93,604 texts tokenized
Run aug=0.0 lr=5e-05 seed=42
Loading weights: 100%|███████████████████████| 290/290 [00:00<00:00, 524.56it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/kaggle/working/src/train_llm.py:156: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
      epoch 1 batch 5

In [6]:
import pandas as pd
t = pd.read_csv("results/tuning/llm_efficiency.csv")
print(t[["lr", "best_epoch", "val_auroc", "train_time_s"]])
print("\nBest learning rate for session 2:", t.loc[t["val_auroc"].idxmax(), "lr"])

        lr  best_epoch  val_auroc  train_time_s
0  0.00005           2   0.999508   4201.372499
1  0.00010           2   0.999668   4192.175534
2  0.00020           2   0.999582   4193.062392

Best learning rate for session 2: 0.0001
